# Turb-DETR Week 2B: Training
**GPU required. Run AFTER Week 2A.**

1. Restore patched Ultralytics
2. Rebuild data (symlinks + augmentation)
3. Sanity check: Turb-DETR on clean (10 epochs)
4. Main: Turb-DETR on augmented (20 epochs)
5. Evaluate all conditions
6. Compare vs baseline

**~4-5 hours total on T4**

In [ ]:
import os, time, shutil
t0 = time.time()
import torch
if not torch.cuda.is_available(): raise RuntimeError("NO GPU")
print(f"GPU: {torch.cuda.get_device_name(0)}")

!pip install -q ultralytics==8.3.40

from google.colab import drive
drive.mount('/content/drive')

import ultralytics
from pathlib import Path

PREP = Path("/content/drive/MyDrive/turb_detr_results/week2_prep")
block_dst = Path(ultralytics.__file__).parent / "nn" / "modules" / "block.py"

if (PREP / "block_patched.py").exists():
    shutil.copy2(PREP / "block_patched.py", block_dst)
    print("Patched block.py restored")
else:
    print("ERROR: Run Week 2A first!")

assert "class SimAM" in block_dst.read_text(), "SimAM not found!"
print("SimAM verified")

DATASET = "/content/drive/MyDrive/underwater_datasets/trash_ICRA19/dataset"
if not os.path.exists(DATASET):
    base = "/content/drive/MyDrive/underwater_datasets"
    for n in os.listdir(base):
        if "icra" in n.lower():
            for sub in ["dataset", ""]:
                cand = os.path.join(base, n, sub) if sub else os.path.join(base, n)
                if os.path.exists(os.path.join(cand, "train")):
                    DATASET = cand; break

os.makedirs("/content/turb-detr", exist_ok=True)
os.chdir("/content/turb-detr")
print(f"Dataset: {DATASET} ({time.time()-t0:.1f}s)")


In [ ]:
import os, subprocess, shutil, cv2, numpy as np, time
from pathlib import Path
from tqdm import tqdm
t0 = time.time()

def jaffe(image, c=0.15, depth=3.0, seed=None):
    rng = np.random.RandomState(seed)
    img = image.astype(np.float32) / 255.0
    att = np.exp(-c * depth)
    bs = np.zeros_like(img)
    bs[:,:,0] = 0.10*(1-att); bs[:,:,1] = 0.30*(1-att); bs[:,:,2] = 0.05*(1-att)
    t = img * att + bs
    n = rng.normal(0, 0.02*c*10, img.shape).astype(np.float32)
    return (np.clip(t+n, 0, 1)*255).astype(np.uint8)

SRC = Path(DATASET); OUT = Path("data/processed"); LOCAL = Path("/content/local_labels")
LEVELS = {"light": 0.05, "medium": 0.15, "heavy": 0.30}

# Restructure + filter ROV
for split in ["train", "val", "test"]:
    sd = SRC/split; (OUT/split/"images").mkdir(parents=True, exist_ok=True)
    for f in sd.glob("*.jpg"):
        d = OUT/split/"images"/f.name
        if not d.exists(): os.symlink(f.resolve(), d)
    ll = LOCAL/split; ll.mkdir(parents=True, exist_ok=True)
    subprocess.run(f"cp {sd}/*.txt {ll}/ 2>/dev/null", shell=True)
    (OUT/split/"labels").mkdir(parents=True, exist_ok=True)
    for txt in ll.glob("*.txt"):
        lines = [l.strip() for l in open(txt) if len(l.strip().split())==5 and l.strip().split()[0]!="2"]
        with open(OUT/split/"labels"/txt.name,"w") as f:
            f.write("\n".join(lines)+"\n" if lines else "")
    print(f"  {split}: structured")

bad = OUT/"train"/"images"/"obj1658_frame0002383 (1).jpg"
if bad.exists() or bad.is_symlink(): os.unlink(bad)

# Augment train
train_imgs = sorted((OUT/"train"/"images").glob("*.jpg"))
print(f"\nAugmenting {len(train_imgs)} train imgs x3...")
for ln, cv in LEVELS.items():
    oi = Path(f"data/augmented_train/{ln}/images"); oi.mkdir(parents=True, exist_ok=True)
    ol = Path(f"data/augmented_train/{ln}/labels"); ol.mkdir(parents=True, exist_ok=True)
    for i, ip in enumerate(tqdm(train_imgs, desc=f"train-{ln}")):
        img = cv2.imread(str(ip))
        if img is None: continue
        cv2.imwrite(str(oi/f"{ln}_{ip.name}"), jaffe(img, c=cv, seed=42+i))
        ls = OUT/"train"/"labels"/f"{ip.stem}.txt"
        if ls.exists(): shutil.copy2(ls, ol/f"{ln}_{ip.stem}.txt")

# Augment test
test_imgs = sorted((OUT/"test"/"images").glob("*.jpg"))
print(f"\nAugmenting {len(test_imgs)} test imgs x3...")
for ln, cv in LEVELS.items():
    oi = Path(f"data/augmented_test/{ln}/images"); oi.mkdir(parents=True, exist_ok=True)
    ol = Path(f"data/augmented_test/{ln}/labels"); ol.mkdir(parents=True, exist_ok=True)
    for i, ip in enumerate(tqdm(test_imgs, desc=f"test-{ln}")):
        img = cv2.imread(str(ip))
        if img is None: continue
        cv2.imwrite(str(oi/ip.name), jaffe(img, c=cv, seed=42+i))
        ls = OUT/"test"/"labels"/f"{ip.stem}.txt"
        if ls.exists(): shutil.copy2(ls, ol/ls.name)

# Combined
CI = Path("data/combined_train/images"); CI.mkdir(parents=True, exist_ok=True)
CL = Path("data/combined_train/labels"); CL.mkdir(parents=True, exist_ok=True)
for f in (OUT/"train"/"images").glob("*.jpg"):
    d = CI/f.name
    if not d.exists(): os.symlink(f.resolve(), d)
for f in (OUT/"train"/"labels").glob("*.txt"):
    d = CL/f.name
    if not d.exists(): os.symlink(f.resolve(), d)
for ln in LEVELS:
    for f in Path(f"data/augmented_train/{ln}/images").glob("*.jpg"):
        d = CI/f.name
        if not d.exists(): os.symlink(f.resolve(), d)
    for f in Path(f"data/augmented_train/{ln}/labels").glob("*.txt"):
        d = CL/f.name
        if not d.exists(): os.symlink(f.resolve(), d)

total = len(list(CI.glob("*.jpg")))
print(f"\nAll data ready ({time.time()-t0:.1f}s) | Combined: {total} imgs")


In [ ]:
import yaml, os
from pathlib import Path
os.makedirs("configs/datasets", exist_ok=True)

with open("configs/datasets/trash_icra19_clean.yaml","w") as f:
    yaml.dump({"path":str(Path("data/processed").resolve()),"train":"train/images","val":"val/images","test":"test/images","names":{0:"plastic",1:"bio"}},f)

with open("configs/datasets/trash_icra19_augmented.yaml","w") as f:
    yaml.dump({"path":str(Path("data").resolve()),"train":"combined_train/images","val":"processed/val/images","test":"processed/test/images","names":{0:"plastic",1:"bio"}},f)

for lv in ["light","medium","heavy"]:
    with open(f"configs/datasets/trash_icra19_turbid_{lv}.yaml","w") as f:
        yaml.dump({"path":str(Path(f"data/augmented_test/{lv}").resolve()),"train":"images","val":"images","test":"images","names":{0:"plastic",1:"bio"}},f)

print("Configs ready")


In [ ]:
import random, numpy as np, torch, os
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED); os.environ["PYTHONHASHSEED"] = str(SEED)
torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
print(f"Seeds = {SEED}")


## Phase 1: Sanity Check (10 epochs on clean)
If mAP drops below 0.85 compared to baseline 0.985, SimAM injection is broken.

In [ ]:
import time
from ultralytics import RTDETR
t0 = time.time()
model = RTDETR("rtdetr-l.pt")
model.train(data="configs/datasets/trash_icra19_clean.yaml", epochs=10, imgsz=640, batch=16,
            seed=42, project="results", name="turbdetr_sanity", device=0, workers=2, patience=5)
print(f"Sanity check done ({(time.time()-t0)/60:.1f} min)")


## Phase 2: Main Training (20 epochs on augmented)
This is THE experiment. ~3 hours on T4.

In [ ]:
import time
from ultralytics import RTDETR
t0 = time.time()
model = RTDETR("rtdetr-l.pt")
model.train(data="configs/datasets/trash_icra19_augmented.yaml", epochs=20, imgsz=640, batch=16,
            seed=42, project="results", name="turbdetr_augmented", device=0, workers=2, patience=10)
print(f"\nTURB-DETR TRAINING COMPLETE ({(time.time()-t0)/60:.1f} min)")


In [ ]:
from pathlib import Path
from ultralytics import RTDETR

BEST = "results/turbdetr_augmented/weights/best.pt"
if not Path(BEST).exists(): BEST = "results/turbdetr_augmented/weights/last.pt"
model = RTDETR(BEST)

print("=== Clean ===")
cr = model.val(data="configs/datasets/trash_icra19_clean.yaml", split="test", device=0)
td_clean = {"map50": cr.box.map50, "map50_95": cr.box.map, "ap": cr.box.ap50}
print(f"  mAP@0.5: {td_clean['map50']:.4f}")

td_turbid = {}
for lv in ["light", "medium", "heavy"]:
    print(f"\n=== {lv} ===")
    r = model.val(data=f"configs/datasets/trash_icra19_turbid_{lv}.yaml", split="test", device=0)
    td_turbid[lv] = {"map50": r.box.map50, "map50_95": r.box.map, "ap": r.box.ap50}
    print(f"  mAP@0.5: {td_turbid[lv]['map50']:.4f}")
print("\nAll evaluations complete")


In [ ]:
import csv, os
os.makedirs("results/tables", exist_ok=True)

BL = {"clean": 0.9853, "light": 0.9735, "medium": 0.7368, "heavy": 0.1494}

print()
print("="*80)
print("  VANILLA RT-DETR vs TURB-DETR")
print("="*80)
print(f"  {'Condition':<16} {'Baseline':>12} {'Turb-DETR':>12} {'Delta':>10}")
print("-"*80)

rows = []
for label, key, val in [("Clean","clean",td_clean), ("Light","light",td_turbid.get("light",{})),
                          ("Medium","medium",td_turbid.get("medium",{})), ("Heavy","heavy",td_turbid.get("heavy",{}))]:
    bl = BL[key]
    td = val.get("map50", 0)
    d = td - bl
    ds = f"+{d:.4f}" if d >= 0 else f"{d:.4f}"
    print(f"  {label:<16} {bl:>12.4f} {td:>12.4f} {ds:>10}")
    rows.append({"condition": key, "baseline": bl, "turbdetr": td, "delta": d})

print("="*80)

# Per-class heavy
print(f"\n  Per-class AP@0.5 at Heavy:")
bl_h = [0.2155, 0.0834]
td_h = td_turbid.get("heavy", {}).get("ap", [0, 0])
for i, nm in enumerate(["plastic", "bio"]):
    b = bl_h[i]; t = td_h[i] if i < len(td_h) else 0
    print(f"    {nm}: {b:.4f} -> {t:.4f} (delta {t-b:+.4f})")

hi = td_turbid.get("heavy",{}).get("map50",0) - BL["heavy"]
print()
if hi > 0.10:
    print(f"  STRONG IMPROVEMENT (+{hi:.4f}). SimAM works.")
elif hi > 0.03:
    print(f"  MODERATE IMPROVEMENT (+{hi:.4f}). Need ablation study.")
else:
    print(f"  MINIMAL IMPROVEMENT (+{hi:.4f}). Check injection.")
print("="*80)

with open("results/tables/baseline_vs_turbdetr.csv","w",newline="") as f:
    w = csv.DictWriter(f, fieldnames=["condition","baseline","turbdetr","delta"])
    w.writeheader(); w.writerows(rows)
print(f"\nSaved: results/tables/baseline_vs_turbdetr.csv")
print("\n*** SEND THIS TABLE TO CLAUDE ***")


In [ ]:
import shutil
from pathlib import Path
SAVE = Path("/content/drive/MyDrive/turb_detr_results/week2")
SAVE.mkdir(parents=True, exist_ok=True)

for run in ["turbdetr_sanity", "turbdetr_augmented"]:
    for w in ["best.pt", "last.pt"]:
        s = Path(f"results/{run}/weights/{w}")
        if s.exists(): shutil.copy2(s, SAVE/f"{run}_{w}"); print(f"  {run}/{w} saved")
    s = Path(f"results/{run}/results.csv")
    if s.exists(): shutil.copy2(s, SAVE/f"{run}_curves.csv")

for f in Path("results/tables").glob("*.csv"):
    shutil.copy2(f, SAVE/f.name)

print(f"All saved to {SAVE}")
